# Cradle: Baby Cry Classification Using Machine Learning

This notebook demonstrates the **machine-learning component** of the Cradle project. It loads labeled audio files, extracts MFCC features, trains a Random Forest classifier, evaluates the model on a held-out test split, and predicts the class of a new audio recording.

**Classes:** `hungry`, `burping`, `discomfort`, `belly_pain`, `tired`

> **Important limitation:** A cry-category prediction is experimental and is not a medical diagnosis. Caregivers should not rely on this model alone to determine a baby's needs.

## 1. Install/import dependencies
Run this notebook in Kaggle or another Python environment with the dataset available.

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_fscore_support
)
from imblearn.over_sampling import SMOTE

## 2. Configure the dataset path
Update `dataset_path` if your dataset is stored somewhere else. Each class should have its own folder.

In [ ]:
dataset_path = "/kaggle/input/datasets/rajeshwarimurugadass/samples/donateacry_corpus"
classes = ["hungry", "burping", "discomfort", "belly_pain", "tired"]

for label in classes:
    folder_path = os.path.join(dataset_path, label)
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Missing class folder: {folder_path}")
    print(f"{label}: {len(os.listdir(folder_path))} files")

## 3. Extract MFCC features
MFCCs (Mel-Frequency Cepstral Coefficients) summarize aspects of the audio spectrum. Averaging each coefficient over time gives one fixed-length feature vector per recording. This example uses the first two seconds, matching the original workflow.

In [ ]:
X = []
y = []
failed_files = []

for label in classes:
    folder_path = os.path.join(dataset_path, label)

    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)
        if not os.path.isfile(file_path):
            continue

        try:
            audio, sr = librosa.load(file_path, sr=None, duration=2.0)
            if audio.size == 0:
                raise ValueError("Audio file is empty")

            mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
            feature = np.mean(mfcc.T, axis=0)

            X.append(feature)
            y.append(label)
        except Exception as exc:
            failed_files.append((file_path, str(exc)))

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y)

print("Feature matrix shape:", X.shape)
print("Label array shape:", y.shape)
print("Samples per class:")
print(pd.Series(y).value_counts().reindex(classes, fill_value=0))
print("Files skipped:", len(failed_files))

if len(X) == 0:
    raise ValueError("No audio files were successfully processed. Check the dataset path and file formats.")

## 4. Split first, then balance training data
The test set must remain untouched by SMOTE. This helps avoid leakage from synthetic samples into the evaluation set. If a class has too few examples for a stratified split or SMOTE, add more real recordings before using this workflow.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples before balancing:", len(y_train))
print("Held-out test samples:", len(y_test))

# Apply SMOTE only to training data. Choose k_neighbors based on the smallest class.
counts = pd.Series(y_train).value_counts()
if counts.min() >= 2:
    k = min(5, int(counts.min()) - 1)
    smote = SMOTE(random_state=42, k_neighbors=k)
    X_train_fit, y_train_fit = smote.fit_resample(X_train, y_train)
    print("Training samples after SMOTE:", len(y_train_fit))
else:
    # Avoid SMOTE failure for a class with fewer than two training examples.
    X_train_fit, y_train_fit = X_train, y_train
    print("SMOTE skipped: at least one class has fewer than 2 training examples.")

## 5. Train the Random Forest Classifier

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_fit, y_train_fit)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Held-out test accuracy: {accuracy:.2%}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, labels=classes, zero_division=0))

## 6. Confusion matrix
Rows represent actual labels and columns represent predicted labels. Off-diagonal values show mistakes.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation=35, colorbar=False)
ax.set_title("Confusion Matrix — Held-out Test Set")
plt.tight_layout()
plt.show()

## 7. Actual vs predicted labels
This plot compares labels for the first 25 test examples. Categories are encoded only for plotting; they are not numerical quantities.

In [ ]:
n_show = min(25, len(y_test))
class_to_num = {label: i for i, label in enumerate(classes)}
actual_num = [class_to_num.get(v, -1) for v in y_test[:n_show]]
pred_num = [class_to_num.get(v, -1) for v in y_pred[:n_show]]

plt.figure(figsize=(12, 5))
plt.plot(range(n_show), actual_num, marker="o", linestyle="-", label="Actual")
plt.plot(range(n_show), pred_num, marker="x", linestyle="--", label="Predicted")
plt.yticks(range(len(classes)), classes)
plt.xlabel("Test sample")
plt.ylabel("Cry category")
plt.title("Actual vs Predicted (first test samples)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Per-class precision, recall, and F1-score

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=classes, zero_division=0
)

metrics_df = pd.DataFrame({
    "class": classes,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "support": support
})
display(metrics_df)

metrics_df.set_index("class")[["precision", "recall", "f1_score"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1)
)
plt.title("Per-class Precision, Recall, and F1-score")
plt.ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print("Weighted precision/recall/F1:")
print(classification_report(y_test, y_pred, labels=classes, zero_division=0))

## 9. Accuracy plot

In [ ]:
plt.figure(figsize=(5, 4))
plt.bar(["Held-out accuracy"], [accuracy * 100])
plt.ylim(0, 100)
plt.ylabel("Accuracy (%)")
plt.title("Random Forest Performance")
plt.tight_layout()
plt.show()

## 10. MFCC feature distribution
This is a histogram of the first MFCC feature across recordings. It is a feature-distribution visualization; it does **not** prove that the data follows a Gaussian distribution.

In [ ]:
plt.figure(figsize=(8, 5))
for label in classes:
    values = X[y == label, 0]
    if len(values):
        plt.hist(values, bins=25, alpha=0.4, label=label, density=True)
plt.title("Distribution of the First MFCC Feature")
plt.xlabel("Mean MFCC coefficient 1")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Spectrogram example

In [ ]:
# Use the first successfully processed audio file from the hungry folder for visualization.
example_folder = os.path.join(dataset_path, "hungry")
example_files = [
    os.path.join(example_folder, f)
    for f in os.listdir(example_folder)
    if os.path.isfile(os.path.join(example_folder, f))
]

if example_files:
    audio, sr = librosa.load(example_files[0], sr=None)
    stft = librosa.stft(audio)
    db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

    plt.figure(figsize=(10, 4))
    librosa.display.specshow(db, sr=sr, x_axis="time", y_axis="hz")
    plt.colorbar(format="%+2.0f dB")
    plt.title("Example Audio Spectrogram")
    plt.tight_layout()
    plt.show()
else:
    print("No example audio found in the hungry folder.")

## 12. Test a new audio recording
Set `new_audio` to the exact path of a new audio file. The MFCC settings must match training. The output is a model prediction, not a diagnosis.

In [ ]:
# Example Kaggle path — replace with the path to your uploaded test audio.
new_audio = "/kaggle/input/datasets/rajeshwarimurugadass/test-audio/testing audio.wav"

if not os.path.isfile(new_audio):
    print("Test audio not found. Update new_audio to the exact path shown under /kaggle/input.")
else:
    test_signal, test_sr = librosa.load(new_audio, sr=None, duration=2.0)
    test_mfcc = librosa.feature.mfcc(y=test_signal, sr=test_sr, n_mfcc=13)
    test_feature = np.mean(test_mfcc.T, axis=0).reshape(1, -1)
    prediction = model.predict(test_feature)[0]
    print("Predicted cry category:", prediction)
    print("Reminder: this is experimental and not a medical diagnosis.")

## Notes and limitations
- The reported test accuracy is calculated on the held-out portion of the dataset and may vary with the data and split.
- SMOTE is applied only to training data; the test data is not resampled.
- MFCC averaging is a simple baseline and may discard useful timing information.
- Dataset labels and audio quality may be noisy or uneven across categories.
- Do not interpret predictions as medical advice or use this model as the sole basis for baby-care decisions.